# HAR Deep Vision — Google Colab Training Notebook

This notebook trains the full HAR model on a larger UCF101 subset using a free Colab GPU (T4).

## Steps:
1. Upload your project zip file (or mount Google Drive)
2. Install dependencies
3. Train the model
4. Download the resulting `best_model.pth` checkpoint
5. Place it in `har-project/checkpoints/best_model.pth` locally

---

**Runtime:** Set Runtime → Change runtime type → T4 GPU


In [ ]:
# Step 1: Check GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Step 2: Install dependencies
!pip install -q mediapipe==0.10.14 pyyaml seaborn scikit-learn

In [ ]:
# Step 3A: Option A — Upload your project zip
from google.colab import files
print('Upload your har-project.zip file...')
# uploaded = files.upload()  # Uncomment to upload

# Then extract:
# !unzip -q har-project.zip -d /content/
# %cd /content/har-project

In [ ]:
# Step 3B: Option B — Download UCF101 subset directly (faster)
# This downloads just the 6 classes used in the project
import os

!mkdir -p /content/har-project/data/ucf101

# Clone just the backend code from a git repo (if you have one)
# Or manually upload the backend/ directory

# Download UCF101 sample subset (6 classes, ~10 videos each)
# This is a small subset for demonstration
CLASSES = ['ApplyEyeMakeup', 'Biking', 'JumpingJack', 'PushUps', 'Squats', 'WalkingWithDog']
BASE_URL = 'https://www.crcv.ucf.edu/THUMOS14/UCF101/UCF101/{cls}/{cls}.zip'
print('NOTE: Download UCF101 from https://www.crcv.ucf.edu/data/UCF101.php')
print('Extract these class folders to /content/har-project/data/ucf101/')
for c in CLASSES:
    print(f'  - {c}/')

In [ ]:
# Step 4: Setup paths — EDIT THIS if your project is in a different location
import sys
PROJECT_ROOT = '/content/har-project'
sys.path.insert(0, f'{PROJECT_ROOT}/backend')

import os
os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

In [ ]:
# Step 5: Configure training
import yaml
from pathlib import Path

with open('backend/config.yaml') as f:
    cfg = yaml.safe_load(f)

# Override paths for Colab
cfg['data']['dataset_path'] = 'data/ucf101'
cfg['data']['class_names_file'] = 'data/class_names.json'
cfg['training']['checkpoint_dir'] = 'checkpoints'
cfg['evaluation']['results_dir'] = 'results'

# Training hyperparameters — tuned for GPU + larger dataset
cfg['training']['batch_size'] = 16
cfg['training']['epochs'] = 50
cfg['training']['learning_rate'] = 0.0001
cfg['training']['scheduler'] = 'cosine'
cfg['training']['mixed_precision'] = True   # use AMP on GPU
cfg['training']['early_stopping_patience'] = 10
cfg['training']['label_smoothing'] = 0.1
cfg['model']['use_pose'] = False  # Set True if MediaPipe is installed and you want pose
cfg['data']['num_workers'] = 2

print('Config loaded. Dataset path:', cfg['data']['dataset_path'])

In [ ]:
# Step 6: Build DataLoaders
from preprocessing.dataset_loader import build_dataloaders

train_loader, val_loader, test_loader, class_names = build_dataloaders(cfg)
print(f'Classes ({len(class_names)}): {class_names}')
print(f'Train: {len(train_loader.dataset)} samples ({len(train_loader)} batches)')
print(f'Val:   {len(val_loader.dataset)} samples')
print(f'Test:  {len(test_loader.dataset)} samples')

In [ ]:
# Step 7: Build and inspect the model
import torch
from models.har_model import HARModel

cfg['model']['num_classes'] = len(class_names)
model = HARModel.from_config(cfg)
print(f'Total parameters: {model.count_parameters():,}')
print(f'Trainable parameters: {model.count_parameters(trainable_only=True):,}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')

In [ ]:
# Step 8: Train the model
from training.trainer import train
from training.evaluator import plot_training_curves

history = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    class_names=class_names,
    cfg=cfg,
    results_dir=cfg['evaluation']['results_dir'],
)

plot_training_curves(history, save_dir=cfg['evaluation']['results_dir'])
print('Training complete!')

In [ ]:
# Step 9: Evaluate on test set
from training.evaluator import evaluate_model

# Load best checkpoint
best_ckpt = Path('checkpoints/best_model.pth')
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Best checkpoint: epoch {ckpt["epoch"]}, val_acc={ckpt["best_val_accuracy"]:.4f}')

model = model.to(device)
metrics = evaluate_model(
    model, test_loader, class_names, device,
    results_dir=cfg['evaluation']['results_dir'],
    top_k_values=[1, 5],
)

print('\n=== Final Results ===')
print(f'Top-1 Accuracy:  {metrics["top_1_accuracy"]:.4f}')
print(f'Macro F1:        {metrics["macro_f1"]:.4f}')
print(f'FPS estimate:    {metrics["fps_estimate"]:.1f}')

In [ ]:
# Step 10: Download the trained checkpoint
from google.colab import files

print('Downloading best_model.pth ...')
files.download('checkpoints/best_model.pth')

print('Downloading metrics and plots ...')
files.download('results/metrics.json')
files.download('results/confusion_matrix.png')
files.download('results/training_curves.png')
files.download('results/validation_curves.png')
files.download('results/classification_report.txt')
files.download('results/training_history.json')

print('\n✅ Download complete!')
print('Place best_model.pth in: har-project/checkpoints/best_model.pth')
print('Place results files in:  har-project/results/')

## After downloading

1. Copy `best_model.pth` → `har-project/checkpoints/best_model.pth` (overwrite)
2. Copy `metrics.json`, `*.png`, etc. → `har-project/results/`
3. Restart the backend: `run_backend.bat`
4. The model will automatically load the new checkpoint on startup
5. Reload via API: `POST http://localhost:8000/api/model/load`
